In [ ]:
%pip install -U huggingface_hub

In [ ]:

import sys
import platform
import torch
import huggingface_hub
import transformers
import pandas
import pyarrow
import ultralytics
import torch

In [ ]:



print("Versión de Python:", sys.version)
print("Sistema operativo:", platform.platform())
print("Versión de PyTorch:", torch.__version__)


if torch.cuda.is_available():
    print("\nGPU disponible:", torch.cuda.get_device_name(0))
    print("Versión CUDA de PyTorch:", torch.version.cuda)

    memoria = torch.cuda.get_device_properties(0).total_memory
    print(f"Memoria total de GPU: {memoria / (1024**3):.2f} GB")
else:
    print("\nNo se detectó una GPU.")
    print("Revisa la configuración del entorno de ejecución de Colab.")

DESCARGAR EL CONJUNTO DE DATOS

In [ ]:

from pathlib import Path
from huggingface_hub import snapshot_download

# Carpeta local de destino
DESTINO = Path("/home/caro/IA Vision/data/odiug77-wildfire")

DESTINO.mkdir(parents=True, exist_ok=True)

ruta_descargada = snapshot_download(
    repo_id="odiug77/wildfire",
    repo_type="dataset",
    local_dir=str(DESTINO)
)

print("Descarga completada.")
print("Ubicación:", ruta_descargada)
print("Existe la carpeta:", DESTINO.exists())


In [ ]:

from pathlib import Path

DESTINO = Path("/home/caro/IA Vision/data/odiug77-wildfire")

print("Archivos principales:")

for archivo in sorted(DESTINO.iterdir()):
    print(archivo.name, "—", "carpeta" if archivo.is_dir() else "archivo")

print("\nArchivos YAML encontrados:")
for archivo in DESTINO.rglob("*.yaml"):
    print(archivo)


In [ ]:

from pathlib import Path

DESTINO = Path("/home/caro/IA Vision/data/odiug77-wildfire")

print("Archivos principales:")

for archivo in sorted(DESTINO.iterdir()):
    print(archivo.name, "—", "carpeta" if archivo.is_dir() else "archivo")

print("\nArchivos YAML encontrados:")
for archivo in DESTINO.rglob("*.yaml"):
    print(archivo)


In [ ]:

from pathlib import Path
from collections import Counter

DESTINO = Path("/home/caro/IA Vision/data/odiug77-wildfire")
DATA = DESTINO / "data"

print("=== RESUMEN DE CARPETAS ===")

for carpeta in sorted(p for p in DATA.rglob("*") if p.is_dir()):
    archivos = [p for p in carpeta.iterdir() if p.is_file()]
    extensiones = Counter(p.suffix.lower() for p in archivos)

    print(f"\n{carpeta.relative_to(DESTINO)}")
    print("  Archivos:", len(archivos))
    print("  Extensiones:", dict(extensiones))

print("\n=== ARCHIVOS DE CLASES ===")

for archivo in DESTINO.rglob("classes.txt"):
    print(f"\nArchivo: {archivo.relative_to(DESTINO)}")
    print(archivo.read_text(encoding="utf-8"))

print("\n=== ARCHIVOS YAML ===")
yamls = list(DESTINO.rglob("*.yaml")) + list(DESTINO.rglob("*.yml"))
print([str(p.relative_to(DESTINO)) for p in yamls] or "No se encontraron")


In [ ]:

from pathlib import Path
from collections import Counter
import yaml

# Dataset original descargado de Hugging Face
DESTINO = Path("/home/caro/IA Vision/data/odiug77-wildfire")

IMAGES_ORIG = DESTINO / "data" / "train" / "images"
LABELS_ORIG = DESTINO / "data" / "train" / "yololabels"
CLASSES_YOLO = LABELS_ORIG / "classes.txt"

# Comprobar las rutas
assert IMAGES_ORIG.is_dir(), f"No existe: {IMAGES_ORIG}"
assert LABELS_ORIG.is_dir(), f"No existe: {LABELS_ORIG}"
assert CLASSES_YOLO.is_file(), f"No existe: {CLASSES_YOLO}"

# Leer el orden de clases específico de las etiquetas YOLO
clases = [
    linea.strip()
    for linea in CLASSES_YOLO.read_text(encoding="utf-8").splitlines()
    if linea.strip()
]

print("Clases de las etiquetas YOLO:", clases)

assert clases == ["smoke", "fire"], (
    "El orden de clases es diferente del esperado. "
    "No continúes con la evaluación hasta revisarlo."
)

# Inspeccionar etiquetas antes de evaluar
conteo_ids = Counter()
archivos_labels = {
    p.stem: p
    for p in LABELS_ORIG.glob("*.txt")
    if p.name != "classes.txt"
}

imagenes = [
    p for p in IMAGES_ORIG.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
]
stems_imagenes = {p.stem for p in imagenes}

malformadas = []
for archivo in archivos_labels.values():
    for numero, linea in enumerate(
        archivo.read_text(encoding="utf-8").splitlines(), start=1
    ):
        if not linea.strip():
            continue

        valores = linea.split()
        if len(valores) != 5:
            malformadas.append((archivo.name, numero, linea))
            continue

        try:
            clase_id = int(valores[0])
            if clase_id not in (0, 1):
                malformadas.append((archivo.name, numero, linea))
                continue

            conteo_ids[clase_id] += 1
        except ValueError:
            malformadas.append((archivo.name, numero, linea))

sin_etiqueta = stems_imagenes - set(archivos_labels)
labels_sin_imagen = set(archivos_labels) - stems_imagenes

print("\n=== RESUMEN ===")
print("Imágenes:", len(imagenes))
print("Archivos de etiquetas:", len(archivos_labels))
print("Objetos por ID:", dict(sorted(conteo_ids.items())))
print("Imágenes sin archivo de etiqueta:", len(sin_etiqueta))
print("Etiquetas sin imagen correspondiente:", len(labels_sin_imagen))
print("Líneas de etiqueta inválidas:", len(malformadas))

if malformadas:
    print("Ejemplos de etiquetas inválidas:", malformadas[:5])
    raise ValueError("Corrige o inspecciona las etiquetas antes de evaluar.")

if labels_sin_imagen:
    raise ValueError("Hay etiquetas que no corresponden a ninguna imagen.")

# Estructura compatible con Ultralytics
EVAL_ROOT = DESTINO.parent / "odiug77-wildfire-eval"
IMAGES_VAL = EVAL_ROOT / "images" / "val"
LABELS_VAL = EVAL_ROOT / "labels" / "val"

for enlace, origen in [
    (IMAGES_VAL, IMAGES_ORIG),
    (LABELS_VAL, LABELS_ORIG),
]:
    enlace.parent.mkdir(parents=True, exist_ok=True)

    if enlace.is_symlink():
        enlace.unlink()
    elif enlace.exists():
        raise FileExistsError(
            f"Ya existe una carpeta real en {enlace}. "
            "No se modificará."
        )

    enlace.symlink_to(origen.resolve(), target_is_directory=True)

# YAML de evaluación
YAML_EVAL = EVAL_ROOT / "data.yaml"

config_eval = {
    "path": str(EVAL_ROOT),
    "train": "images/val",
    "val": "images/val",
    "test": "images/val",
    "nc": 2,
    "names": {
        0: "smoke",
        1: "fire",
    },
}

with YAML_EVAL.open("w", encoding="utf-8") as f:
    yaml.safe_dump(config_eval, f, sort_keys=False)

print("\n=== CONFIGURACIÓN CREADA ===")
print("YAML:", YAML_EVAL)
print("Imágenes accesibles:", IMAGES_VAL.is_dir())
print("Etiquetas accesibles:", LABELS_VAL.is_dir())
print(YAML_EVAL.read_text(encoding="utf-8"))


Descargar el modelo

In [ ]:

from huggingface_hub import hf_hub_download
from ultralytics import YOLO
from pathlib import Path
import yaml

repositorio2 = "rabahdev/fire-smoke-yolov8n"

ruta_modelo2 = hf_hub_download(
    repo_id=repositorio2,
    filename="best.pt"
)

modelo2 = YOLO(ruta_modelo2)

print("Modelo 2 cargado:", modelo2.names)
print("GPU disponible:", torch.cuda.is_available())

In [ ]:

from pathlib import Path
import yaml

# ============================================================
# 1. RUTAS LOCALES EN WSL2 / UBUNTU
# ============================================================

DESTINO = Path(
    "/home/caro/IA Vision/data/odiug77-wildfire"
)

# Imágenes y etiquetas originales de Hugging Face
carpeta_img = DESTINO / "data" / "train" / "images"
carpeta_lbl = DESTINO / "data" / "train" / "yololabels"

# Archivo que define los IDs de las etiquetas YOLO
archivo_clases = carpeta_lbl / "classes.txt"

# Carpeta donde prepararemos los datos para modelo2
raiz_eval2 = DESTINO.parent / "odiug77-eval-model2"
carpeta_img2 = raiz_eval2 / "images"
carpeta_lbl2 = raiz_eval2 / "labels"

# Comprobar las rutas originales
assert carpeta_img.is_dir(), f"No existe: {carpeta_img}"
assert carpeta_lbl.is_dir(), f"No existe: {carpeta_lbl}"
assert archivo_clases.is_file(), f"No existe: {archivo_clases}"

assert "modelo2" in globals(), (
    "No está cargada la variable modelo2. "
    "Carga el modelo antes de ejecutar esta celda."
)

# Crear carpetas de destino
carpeta_img2.mkdir(parents=True, exist_ok=True)
carpeta_lbl2.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. IDENTIFICAR LAS CLASES DE ORIGEN Y DEL MODELO
# ============================================================

clases_origen = [
    linea.strip().lower()
    for linea in archivo_clases.read_text(
        encoding="utf-8"
    ).splitlines()
    if linea.strip()
]

clases_modelo = {
    int(k): str(v).strip().lower()
    for k, v in modelo2.names.items()
}

print("Clases del dataset:", clases_origen)
print("Clases del modelo:", clases_modelo)

# Crear correspondencia entre los IDs según el nombre
id_modelo_por_nombre = {
    nombre: clase_id
    for clase_id, nombre in clases_modelo.items()
}

if len(clases_origen) != len(clases_modelo):
    raise ValueError(
        "El dataset y el modelo tienen distinto número de clases."
    )

if set(clases_origen) != set(clases_modelo.values()):
    raise ValueError(
        "Los nombres de las clases no coinciden. "
        "Revisa classes.txt y modelo2.names."
    )

conversion2 = {
    id_origen: id_modelo_por_nombre[nombre]
    for id_origen, nombre in enumerate(clases_origen)
}

print("Conversión de IDs:", conversion2)

# Con el dataset descargado esperamos:
# 0 = smoke -> 0 = smoke
# 1 = fire  -> 1 = fire
# Es decir: {0: 0, 1: 1}

# ============================================================
# 3. PREPARAR IMÁGENES Y ETIQUETAS
# ============================================================

extensiones = {".jpg", ".jpeg", ".png", ".webp"}

imagenes = sorted(
    p for p in carpeta_img.iterdir()
    if p.is_file() and p.suffix.lower() in extensiones
)

procesadas = 0
sin_etiqueta = 0
objetos_por_clase = {i: 0 for i in clases_modelo}
archivos_generados = set()

for imagen in imagenes:
    etiqueta = carpeta_lbl / f"{imagen.stem}.txt"

    # Mantener el mismo criterio de tu código de Colab:
    # evaluar imágenes que tengan archivo de etiquetas.
    if not etiqueta.is_file():
        sin_etiqueta += 1
        continue

    lineas_nuevas = []

    for numero, linea in enumerate(
        etiqueta.read_text(encoding="utf-8").splitlines(),
        start=1
    ):
        if not linea.strip():
            continue

        partes = linea.split()

        if len(partes) != 5:
            raise ValueError(
                f"Formato inesperado en {etiqueta}, "
                f"línea {numero}: {linea}"
            )

        try:
            clase = int(partes[0])
        except ValueError as exc:
            raise ValueError(
                f"ID no numérico en {etiqueta}, línea {numero}"
            ) from exc

        if clase not in conversion2:
            raise ValueError(
                f"Clase inesperada {clase} en {etiqueta}, "
                f"línea {numero}"
            )

        nueva_clase = conversion2[clase]
        partes[0] = str(nueva_clase)

        lineas_nuevas.append(" ".join(partes))
        objetos_por_clase[nueva_clase] += 1

    # Crear acceso simbólico a la imagen original.
    # Si se vuelve a ejecutar, reemplazar únicamente el enlace.
    destino_img = carpeta_img2 / imagen.name

    if destino_img.is_symlink():
        destino_img.unlink()
    elif destino_img.exists():
        raise FileExistsError(
            f"Ya existe un archivo que no es enlace: {destino_img}"
        )

    destino_img.symlink_to(imagen.resolve())

    # Crear etiqueta convertida
    destino_lbl = carpeta_lbl2 / f"{imagen.stem}.txt"

    destino_lbl.write_text(
        "\n".join(lineas_nuevas) + (
            "\n" if lineas_nuevas else ""
        ),
        encoding="utf-8"
    )

    archivos_generados.add(destino_lbl.name)
    procesadas += 1

# ============================================================
# 4. CREAR YAML PARA ULTRALYTICS
# ============================================================

config2 = {
    "path": str(raiz_eval2),
    "train": "images",
    "val": "images",
    "test": "images",
    "nc": len(clases_modelo),
    "names": clases_modelo
}

ruta_yaml2 = raiz_eval2 / "dataset.yaml"

ruta_yaml2.write_text(
    yaml.safe_dump(
        config2,
        sort_keys=False,
        allow_unicode=True
    ),
    encoding="utf-8"
)

# ============================================================
# 5. RESUMEN
# ============================================================

print("\n=== PREPARACIÓN COMPLETADA ===")
print("Imágenes originales:", len(imagenes))
print("Imágenes preparadas:", procesadas)
print("Imágenes omitidas por no tener etiquetas:", sin_etiqueta)
print("Etiquetas creadas:", len(archivos_generados))
print("Objetos por ID del modelo:", objetos_por_clase)
print("Directorio:", raiz_eval2)
print("Configuración YAML:", ruta_yaml2)

print("\nContenido del YAML:")
print(ruta_yaml2.read_text(encoding="utf-8"))


In [ ]:

metrics_wildfire = modelo2.val(
    data=str(ruta_yaml2),
    split="val",
    imgsz=640,
    batch=8,
    device=0,
    plots=True,
    verbose=True
)

print("\n=== RESULTADOS: ODIUG77/WILDFIRE ===")

print(f"Precision: {metrics_wildfire.box.mp:.4f}")
print(f"Recall:    {metrics_wildfire.box.mr:.4f}")
print(f"mAP@50:    {metrics_wildfire.box.map50:.4f}")
print(f"mAP@50-95: {metrics_wildfire.box.map:.4f}")

print("\n=== MÉTRICAS POR CLASE ===")

precision_clase = metrics_wildfire.box.p
recall_clase = metrics_wildfire.box.r
map50_clase = metrics_wildfire.box.ap50

if len(precision_clase) == 0:
    print(
        "No se generaron métricas por clase. "
        "Revisa los avisos de Ultralytics y las rutas de etiquetas."
    )
else:
    for i, nombre in modelo2.names.items():
        if i >= len(precision_clase):
            print(f"Sin métricas disponibles para {nombre}")
            continue

        print(f"\nClase: {nombre}")
        print(f"Precision: {precision_clase[i]:.4f}")
        print(f"Recall:    {recall_clase[i]:.4f}")
        print(f"mAP@50:    {map50_clase[i]:.4f}")


In [ ]:
import numpy as np
import pandas as pd

# Reutilizar los resultados de la evaluación anterior.
# Si la variable ya existe, NO vuelve a evaluar el modelo.
try:
    resultados = metrics_wildfire
except NameError:
    print("No se encontró metrics_wildfire. Ejecutando validación...")
    resultados = modelo2.val(
        data=str(ruta_yaml2),
        split="val",
        imgsz=640,
        batch=8,
        device=0,
        plots=True,
        verbose=False
    )

# Extraer la matriz de confusión
cm = resultados.confusion_matrix
matriz = np.asarray(cm.matrix)

# Obtener los nombres de las clases
nombres_modelo = modelo2.names

if isinstance(nombres_modelo, dict):
    clases = [nombres_modelo[i] for i in sorted(nombres_modelo)]
else:
    clases = list(nombres_modelo)

# En detección, la última fila y columna corresponden al fondo
if matriz.shape[0] == len(clases) + 1:
    clases_tabla = clases + ["fondo"]
else:
    clases_tabla = clases

# Mostrar conteos absolutos
tabla = pd.DataFrame(
    matriz.astype(int),
    index=[f"Predicción: {c}" for c in clases_tabla],
    columns=[f"Real: {c}" for c in clases_tabla]
)

print("\n========== MATRIZ DE CONFUSIÓN: CONTEOS ==========")
print(tabla.to_string())

# Normalizar por clase real: cada columna suma aproximadamente 100%
sumas = matriz.sum(axis=0, keepdims=True)

matriz_pct = np.divide(
    matriz * 100.0,
    sumas,
    out=np.zeros_like(matriz, dtype=float),
    where=sumas != 0
)

tabla_pct = pd.DataFrame(
    matriz_pct,
    index=[f"Predicción: {c}" for c in clases_tabla],
    columns=[f"Real: {c}" for c in clases_tabla]
)

print("\n========== MATRIZ NORMALIZADA: PORCENTAJE ==========")
print(tabla_pct.round(2).to_string())

print("\nClases del modelo:", nombres_modelo)
print("Dimensiones de la matriz:", matriz.shape)

In [ ]:
import pandas as pd

umbrales = [0.25, 0.10, 0.05]

resultados_umbral = []

for conf in umbrales:
    print(f"\nEvaluando con confidence = {conf}")

    r = modelo2.val(
        data=str(ruta_yaml2),
        split="val",
        imgsz=640,
        batch=8,
        device=0,
        conf=conf,
        plots=False,
        verbose=False,
        project="/home/caro/IA Vision/runs/detect",
        name=f"umbral_{str(conf).replace('.', '_')}",
        exist_ok=True
    )

    nombres = modelo2.names

    for i, nombre in nombres.items():
        resultados_umbral.append({
            "Umbral": conf,
            "Clase": nombre,
            "Precision": float(r.box.p[i]),
            "Recall": float(r.box.r[i]),
            "mAP50": float(r.box.ap50[i]),
            "mAP50-95": float(r.box.ap[i]),
        })

tabla_umbrales = pd.DataFrame(resultados_umbral)

print("\n========== COMPARACIÓN POR UMBRAL ==========")
print(tabla_umbrales.round(4).to_string(index=False))

In [ ]:
# Evaluación a mayor resolución
# No especificamos conf para mantener la configuración predeterminada.

resultados_960 = modelo2.val(
    data=str(ruta_yaml2),
    split="val",
    imgsz=960,
    batch=4,
    device=0,
    plots=True,
    verbose=True,
    project="/home/caro/IA Vision/runs/detect",
    name="val-imgsz960",
    exist_ok=True
)

print("\n========== RESULTADOS A 960x960 ==========")
print(f"Precisión global: {resultados_960.box.mp:.4f}")
print(f"Recall global:    {resultados_960.box.mr:.4f}")
print(f"mAP@50:           {resultados_960.box.map50:.4f}")
print(f"mAP@50-95:        {resultados_960.box.map:.4f}")

print("\n========== RESULTADOS POR CLASE ==========")

for i, nombre in modelo2.names.items():
    print(f"\nClase: {nombre}")
    print(f"Precision: {resultados_960.box.p[i]:.4f}")
    print(f"Recall:    {resultados_960.box.r[i]:.4f}")
    print(f"mAP@50:    {resultados_960.box.ap50[i]:.4f}")
    print(f"mAP@50-95: {resultados_960.box.ap[i]:.4f}")

print("\nArchivos guardados en:")
print(resultados_960.save_dir)

In [ ]:
from pathlib import Path
import shutil
import numpy as np

# --------------------------------------------------
# 1. RUTAS
# --------------------------------------------------
BASE = Path.home() / "IA Vision"

FUENTE = BASE / "data" / "odiug77-eval-model2"
DESTINO = BASE / "data" / "odiug77-yolov8s-split"

CARPETA_IMAGENES = FUENTE / "images"
CARPETA_ETIQUETAS = FUENTE / "labels"

# Comprobar que los datos existen
assert CARPETA_IMAGENES.exists(), f"No existe: {CARPETA_IMAGENES}"
assert CARPETA_ETIQUETAS.exists(), f"No existe: {CARPETA_ETIQUETAS}"

EXTENSIONES = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

imagenes = sorted(
    p for p in CARPETA_IMAGENES.iterdir()
    if p.is_file() and p.suffix.lower() in EXTENSIONES
)

# --------------------------------------------------
# 2. EMPAREJAR IMÁGENES CON SUS ETIQUETAS
# --------------------------------------------------
pares = []
sin_etiqueta = []

for imagen in imagenes:
    etiqueta = CARPETA_ETIQUETAS / f"{imagen.stem}.txt"

    if not etiqueta.exists():
        sin_etiqueta.append(imagen.name)
        continue

    # ID 0 = smoke, ID 1 = fire
    tiene_fuego = False

    for linea in etiqueta.read_text(encoding="utf-8").splitlines():
        partes = linea.split()

        if not partes:
            continue

        try:
            clase = int(float(partes[0]))
        except ValueError:
            continue

        if clase == 1:
            tiene_fuego = True
            break

    pares.append((imagen, etiqueta, tiene_fuego))

if len(pares) < 10:
    raise ValueError(
        f"Solo se encontraron {len(pares)} pares imagen-etiqueta. "
        "Revisa las rutas antes de continuar."
    )

# --------------------------------------------------
# 3. DIVISIÓN REPRODUCIBLE 80/20
# --------------------------------------------------
rng = np.random.default_rng(42)

train_items = []
val_items = []

# Estratificamos por presencia de fuego en cada imagen
for tiene_fuego in (False, True):
    grupo = [p for p in pares if p[2] == tiene_fuego]

    indices = rng.permutation(len(grupo))
    grupo = [grupo[i] for i in indices]

    if not grupo:
        continue

    n_val = max(1, round(len(grupo) * 0.20))

    val_items.extend(grupo[:n_val])
    train_items.extend(grupo[n_val:])

# --------------------------------------------------
# 4. CREAR CARPETAS DESTINO
# --------------------------------------------------
# Esta carpeta es exclusivamente la nueva división generada.
# No apunta al dataset original.
if DESTINO.exists():
    shutil.rmtree(DESTINO)

for split in ("train", "val"):
    (DESTINO / "images" / split).mkdir(
        parents=True, exist_ok=True
    )
    (DESTINO / "labels" / split).mkdir(
        parents=True, exist_ok=True
    )

# Se crean enlaces simbólicos para ahorrar espacio en disco.
# Las etiquetas se copian a sus nuevas ubicaciones.
for split, elementos in (
    ("train", train_items),
    ("val", val_items),
):
    for imagen, etiqueta, tiene_fuego in elementos:
        destino_img = DESTINO / "images" / split / imagen.name
        destino_lbl = DESTINO / "labels" / split / etiqueta.name

        destino_img.symlink_to(imagen.resolve())
        shutil.copy2(etiqueta, destino_lbl)

# --------------------------------------------------
# 5. CREAR YAML DE ULTRALYTICS
# --------------------------------------------------
ruta_yaml_s = DESTINO / "data.yaml"

ruta_yaml_s.write_text(
    f'path: "{DESTINO.as_posix()}"\n'
    "train: images/train\n"
    "val: images/val\n"
    "nc: 2\n"
    "names:\n"
    "  0: smoke\n"
    "  1: fire\n",
    encoding="utf-8"
)

# --------------------------------------------------
# 6. RESUMEN
# --------------------------------------------------
print("========== DATASET PREPARADO ==========")
print(f"Imágenes emparejadas: {len(pares)}")
print(f"Entrenamiento:        {len(train_items)}")
print(f"Validación:           {len(val_items)}")
print(f"Sin etiqueta:         {len(sin_etiqueta)}")

print(
    "Imágenes con fuego en entrenamiento:",
    sum(item[2] for item in train_items)
)
print(
    "Imágenes con fuego en validación:",
    sum(item[2] for item in val_items)
)

print(f"\nYAML creado: {ruta_yaml_s}")
print(ruta_yaml_s.read_text(encoding="utf-8"))

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import torch

# Verificar GPU
print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Cargar modelo oficial preentrenado
modelo_s = YOLO("yolov8s.pt")

print("\nModelo inicial:", modelo_s.names)
print("Iniciando entrenamiento...")

resultados_train_s = modelo_s.train(
    data=str(ruta_yaml_s),

    epochs=50,
    imgsz=640,
    batch=8,

    device=0,
    workers=4,

    pretrained=True,
    patience=12,
    seed=42,
    deterministic=True,

    cache=False,
    plots=True,
    save=True,
    verbose=True,

    project=str(BASE / "runs" / "detect"),
    name="yolov8s-odiug77-split"
)

# Obtener la ruta de los mejores pesos
ruta_best_s = Path(modelo_s.trainer.best)

print("\n========== ENTRENAMIENTO TERMINADO ==========")
print("Mejores pesos:", ruta_best_s)
print("¿Existen?:", ruta_best_s.exists())
print("Directorio de resultados:", modelo_s.trainer.save_dir)

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import pandas as pd

BASE = Path.home() / "IA Vision"

# Localizar los mejores pesos de YOLOv8s
if "ruta_best_s" in globals() and Path(ruta_best_s).exists():
    ruta_s = Path(ruta_best_s)
else:
    ruta_s = (
        BASE / "runs" / "detect" /
        "yolov8s-odiug77-split" / "weights" / "best.pt"
    )

# Pesos de YOLOv8n especializado
ruta_n = (
    BASE / "models" /
    "rabahdev-fire-smoke-yolov8n" / "best.pt"
)

assert ruta_s.exists(), f"No se encontraron los pesos YOLOv8s: {ruta_s}"
assert ruta_n.exists(), f"No se encontraron los pesos YOLOv8n: {ruta_n}"

# Usar el YAML de la separación train/val 80/20
print("Dataset:", ruta_yaml_s)
print("YOLOv8s:", ruta_s)
print("YOLOv8n:", ruta_n)

modelos = [
    ("YOLOv8n rabahdev", YOLO(str(ruta_n))),
    ("YOLOv8s entrenado", YOLO(str(ruta_s))),
]

filas = []

for nombre, modelo in modelos:
    print(f"\nEvaluando {nombre}...")

    metricas = modelo.val(
        data=str(ruta_yaml_s),
        split="val",
        imgsz=640,
        batch=8,
        device=0,
        plots=False,
        verbose=False
    )

    for i, clase in enumerate(["smoke", "fire"]):
        filas.append({
            "Modelo": nombre,
            "Clase": clase,
            "Precision": float(metricas.box.p[i]),
            "Recall": float(metricas.box.r[i]),
            "mAP50": float(metricas.box.ap50[i]),
            "mAP50-95": float(metricas.box.ap[i]),
            "Inferencia_ms": float(metricas.speed["inference"]),
        })

tabla_comparacion = pd.DataFrame(filas)

print("\n========== COMPARACIÓN EN EL MISMO CONJUNTO ==========")
print(tabla_comparacion.round(4).to_string(index=False))

Entrenar 

In [ ]:
from pathlib import Path
from ultralytics import YOLO

BASE = Path.home() / "IA Vision"

# Modelo original de Rabahdev; conservaremos estos pesos intactos.
ruta_rabahdev = (
    BASE / "models" / "rabahdev-fire-smoke-yolov8n" / "best.pt"
)

# Dataset con la división 80/20 usada para entrenar YOLOv8s.
ruta_yaml_s = (
    BASE / "data" / "odiug77-yolov8s-split" / "data.yaml"
)

assert ruta_rabahdev.exists(), f"No existe el modelo: {ruta_rabahdev}"
assert ruta_yaml_s.exists(), f"No existe el YAML: {ruta_yaml_s}"

modelo_rabahdev_ft = YOLO(str(ruta_rabahdev))

nombres = [
    modelo_rabahdev_ft.names[i]
    for i in sorted(modelo_rabahdev_ft.names)
]

print("Modelo:", ruta_rabahdev)
print("Clases:", modelo_rabahdev_ft.names)
print("\nConfiguración del dataset:")
print(ruta_yaml_s.read_text(encoding="utf-8"))

assert nombres == ["smoke", "fire"], (
    f"El orden de clases no coincide: {nombres}"
)

print("\nVerificación correcta. Podemos iniciar el entrenamiento.")

In [ ]:
import torch
from pathlib import Path

print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

resultados_train_rabahdev = modelo_rabahdev_ft.train(
    data=str(ruta_yaml_s),

    # Mismos parámetros principales que YOLOv8s
    epochs=50,
    imgsz=640,
    batch=8,
    device=0,
    workers=4,

    # Reproducibilidad y parada anticipada
    seed=42,
    deterministic=True,
    patience=12,

    # Guardado y seguimiento
    cache=False,
    plots=True,
    save=True,
    verbose=True,

    # Carpeta distinta para no sobrescribir otros experimentos
    project=str(BASE / "runs" / "detect"),
    name="rabahdev-wildfire-finetune"
)

ruta_best_rabahdev_ft = Path(
    modelo_rabahdev_ft.trainer.best
)

print("\n========== ENTRENAMIENTO FINALIZADO ==========")
print("Mejores pesos:", ruta_best_rabahdev_ft)
print("¿Existe best.pt?:", ruta_best_rabahdev_ft.exists())
print("Resultados:", modelo_rabahdev_ft.trainer.save_dir)

In [ ]:
from pathlib import Path
from ultralytics import YOLO
import pandas as pd

# ============================================
# 1. RUTAS
# ============================================
BASE = Path.home() / "IA Vision"

# Utilizar la ruta obtenida durante el entrenamiento.
if (
    "ruta_best_rabahdev_ft" in globals()
    and Path(ruta_best_rabahdev_ft).exists()
):
    ruta_modelo = Path(ruta_best_rabahdev_ft)
else:
    ruta_modelo = (
        BASE / "runs" / "detect"
        / "rabahdev-wildfire-finetune"
        / "weights" / "best.pt"
    )

# Mismo conjunto de entrenamiento/validación de YOLOv8s
ruta_yaml = (
    BASE / "data" / "odiug77-yolov8s-split" / "data.yaml"
)

assert ruta_modelo.exists(), (
    f"No se encontró best.pt en: {ruta_modelo}"
)
assert ruta_yaml.exists(), (
    f"No se encontró el dataset YAML en: {ruta_yaml}"
)

print("Modelo evaluado:", ruta_modelo)
print("Dataset:", ruta_yaml)

# ============================================
# 2. CARGAR EL MODELO AJUSTADO
# ============================================
modelo_rabahdev_ft = YOLO(str(ruta_modelo))

print("Clases del modelo:", modelo_rabahdev_ft.names)

# ============================================
# 3. EJECUTAR VALIDACIÓN
# ============================================
metricas_ft = modelo_rabahdev_ft.val(
    data=str(ruta_yaml),
    split="val",
    imgsz=640,
    batch=4,
    device=0,
    plots=True,
    verbose=True,
    project=str(BASE / "runs" / "detect"),
    name="eval-rabahdev-finetune",
    exist_ok=True
)

# ============================================
# 4. MÉTRICAS GLOBALES
# ============================================
print("\n========== MÉTRICAS GLOBALES ==========")
print(f"Precision:  {metricas_ft.box.mp:.4f}")
print(f"Recall:     {metricas_ft.box.mr:.4f}")
print(f"mAP@50:     {metricas_ft.box.map50:.4f}")
print(f"mAP@50-95:  {metricas_ft.box.map:.4f}")

# ============================================
# 5. MÉTRICAS POR CLASE
# ============================================
nombres = modelo_rabahdev_ft.names

filas = []

for i in sorted(nombres):
    filas.append({
        "Clase": nombres[i],
        "Precision": float(metricas_ft.box.p[i]),
        "Recall": float(metricas_ft.box.r[i]),
        "mAP50": float(metricas_ft.box.ap50[i]),
        "mAP50-95": float(metricas_ft.box.ap[i]),
    })

tabla_metricas_ft = pd.DataFrame(filas)

print("\n========== MÉTRICAS POR CLASE ==========")
print(tabla_metricas_ft.round(4).to_string(index=False))

# ============================================
# 6. VELOCIDAD
# ============================================
print("\n========== VELOCIDAD ==========")
print(
    f"Preprocesamiento: "
    f"{metricas_ft.speed['preprocess']:.2f} ms/imagen"
)
print(
    f"Inferencia: "
    f"{metricas_ft.speed['inference']:.2f} ms/imagen"
)
print(
    f"Postprocesamiento: "
    f"{metricas_ft.speed['postprocess']:.2f} ms/imagen"
)

print("\nResultados y gráficas:")
print(metricas_ft.save_dir)

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd

BASE = Path.home() / "IA Vision"

ruta_original = (
    BASE / "models" / "rabahdev-fire-smoke-yolov8n" / "best.pt"
)

ruta_yolov8s = (
    BASE / "runs" / "detect"
    / "yolov8s-odiug77-split"
    / "weights" / "best.pt"
)

modelos = [
    ("YOLOv8n original", ruta_original),
    ("YOLOv8n fine-tuning", ruta_modelo),
    ("YOLOv8s entrenado", ruta_yolov8s),
]

filas = []

for nombre, ruta in modelos:
    if not ruta.exists():
        print(f"No se encuentra {nombre}: {ruta}")
        continue

    modelo = YOLO(str(ruta))

    print(f"\nEvaluando {nombre}...")

    r = modelo.val(
        data=str(ruta_yaml),
        split="val",
        imgsz=640,
        batch=4,
        device=0,
        plots=False,
        verbose=False
    )

    filas.append({
        "Modelo": nombre,
        "Precision": float(r.box.mp),
        "Recall": float(r.box.mr),
        "mAP50": float(r.box.map50),
        "mAP50-95": float(r.box.map),
        "Recall_smoke": float(r.box.r[0]),
        "Recall_fire": float(r.box.r[1]),
        "Inferencia_ms": float(r.speed["inference"]),
    })

comparacion_ft = pd.DataFrame(filas)

print("\n========== COMPARACIÓN DE MODELOS ==========")
print(comparacion_ft.round(4).to_string(index=False))